# movetocom_var — the variational centre-of-mass fix

Probes reb_simulation_move_to_com with MEGNO variational particles. An audit found the first-order shift summed the wrong particle array, which silently changed every MEGNO/Lyapunov result. Now fixed and verified bit-identical against C.

This notebook is self-contained: it builds the example with cargo, runs it, and shows the result. Everything it does can also be done by hand in a terminal:

```
cd rebound_rust
cargo build --release --example movetocom_var
cd porttest
..\target\release\examples\movetocom_var.exe
```

In [1]:
import os, subprocess
NB_DIR  = os.getcwd()                       # <crate>/notebooks
ROOT    = os.path.dirname(os.path.dirname(NB_DIR))
CRATE   = os.path.join(ROOT, "rebound_rust")
WORK    = os.path.join(CRATE, "porttest")
if not os.path.exists(os.path.join(CRATE, "Cargo.toml")):
    raise SystemExit(
        "Could not find the crate. Run this notebook from "
        "the notebooks folder of a full checkout: " + CRATE)
# The example that is BUILT and RUN. It is usually the one the
# notebook is named after; where it differs (the stock
# shearing_sheet integrates forever by design) the terminating
# variant is used instead, and the note above says so.
EXAMPLE = "movetocom_var"
OUTFILE = None
os.makedirs(WORK, exist_ok=True)
res = subprocess.run(["cargo", "build", "--release", "--example", EXAMPLE],
                     cwd=CRATE, capture_output=True, text=True)
print(res.stderr.strip()[-400:] or "build ok")

    Finished `release` profile [optimized] target(s) in 0.01s


In [2]:
import struct

def unbits(h):
    """Turn a 16-hex-digit IEEE-754 bit pattern back into a float."""
    return struct.unpack("<d", int(h, 16).to_bytes(8, "little"))[0]

def read_state(path):
    """Read one of the raw-bit state dumps into {label: [floats]}."""
    out = {}
    with open(path) as fh:
        for line in fh:
            parts = line.split()
            if not parts:
                continue
            key, rest = parts[0], parts[1:]
            vals = []
            for tok in rest:
                if len(tok) == 16:
                    try:
                        vals.append(unbits(tok))
                        continue
                    except ValueError:
                        pass
                vals.append(tok)
            out.setdefault(key, []).append(vals)
    return out

def compare(a, b, label_a="C", label_b="Rust"):
    """Byte-compare two dump files and report."""
    ta = open(a, "rb").read().replace(b"\r\n", b"\n")
    tb = open(b, "rb").read().replace(b"\r\n", b"\n")
    if ta == tb:
        print(f"BIT-IDENTICAL: {label_a} and {label_b} agree on every bit")
        return True
    print(f"MISMATCH between {label_a} and {label_b}")
    la, lb = ta.decode().splitlines(), tb.decode().splitlines()
    for i, (x, y) in enumerate(zip(la, lb)):
        if x != y:
            print(f"  line {i}:\n    {label_a}: {x}\n    {label_b}: {y}")
    return False


In [3]:
exe = os.path.join(CRATE, "target", "release", "examples", EXAMPLE + ".exe")
res = subprocess.run([exe], cwd=WORK, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:", res.stderr[-2000:])

com.m 3ff003e8f2d96218 com.x 3f7351b0cb646c7f
movetocom_var_rust done



In [4]:
c  = os.path.join(WORK, "movetocom_var_c.txt")
rs = os.path.join(WORK, "movetocom_var_rust.txt")
if os.path.exists(rs):
    print(open(rs).read())
if os.path.exists(c):
    compare(c, rs)


N 2 N_var 2
p 0 bf7351b0cb646c7f 0000000000000000 0000000000000000 0000000000000000 bf3ccbdb3cdfee05 0000000000000000
p 1 4013c39666a9d82c 0000000000000000 0000000000000000 0000000000000000 3fdd75a0722210c5 0000000000000000
v 0 3f743a2bf4e17c80 3f41605fc561d000 bf3ec6554baafc00 3f2649d3a297c000 3f326e64a5d8f780 bf49a0a36c29e400
v 1 bfcd202b0c6e38d4 bfe1c6d16b520d58 3fdf7bc4778b8fe6 bfc6cd3aab0fdb3d 3fc5438a66da0dd0 3fea37ba0c28d9f9

BIT-IDENTICAL: C and Rust agree on every bit
